In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

print("Initializing industrial environment simulation...")

# 1. Set configuration settings
np.random.seed(42)
random.seed(42)
num_machines = 50
days_of_data = 30  # Let's generate 30 days of minute-by-minute logs

# Start time for our sensor history
start_date = datetime(2026, 6, 1, 0, 0, 0)
total_minutes = days_of_data * 24 * 60

# 2. Build our static factory machinery inventory
machines = []
machine_types = ["CNC Milling Machine", "Industrial Robotic Arm", "Heavy Duty Compressor", "High-Speed Turbine"]

for m_id in range(101, 101 + num_machines):
    machines.append({
        "machine_id": m_id,
        "machine_type": random.choice(machine_types),
        "factory_floor": random.choice(["Floor A", "Floor B", "Floor C"]),
        "installation_year": random.randint(2018, 2025)
    })

df_inventory = pd.DataFrame(machines)

# 3. Generate high-frequency time-series sensor logs
print(f"Generating minute-by-minute data for {num_machines} machines. This creates ~2.1 million rows...")

sensor_logs = []

# Generate timestamps array
timestamps = [start_date + timedelta(minutes=i) for i in range(0, total_minutes, 60)] # Sampling every hour to keep it fast in Colab

log_id = 1
for machine in machines:
    m_id = machine["machine_id"]
    m_type = machine["machine_type"]

    # Establish base operating metrics based on machine type
    if m_type == "High-Speed Turbine":
        base_temp, base_vibe, base_rpm = 75.0, 1.2, 3000
    elif m_type == "Heavy Duty Compressor":
        base_temp, base_vibe, base_rpm = 65.0, 2.8, 1500
    else:
        base_temp, base_vibe, base_rpm = 50.0, 1.5, 1200

    for ts in timestamps:
        # Introduce natural operating noise/fluctuation using normal distribution
        temperature = base_temp + np.random.normal(0, 2.5)
        vibration = base_vibe + np.random.normal(0, 0.3)
        rpm = base_rpm + np.random.normal(0, 20)

        # Inject an intentional anomaly / failure event (e.g., machine overheating randomly)
        status = "Normal"
        if random.random() < 0.002: # 0.2% chance of a severe anomaly spike
            temperature += 35.0
            vibration += 4.5
            rpm *= 0.4 # Machine slows down significantly due to friction/failure
            status = "Anomaly detected"

        sensor_logs.append({
            "log_id": log_id,
            "machine_id": m_id,
            "timestamp": ts.strftime("%Y-%m-%d %H:%M:%S"),
            "temperature_celsius": round(temperature, 2),
            "vibration_mm_s": round(vibration, 3),
            "rotation_speed_rpm": round(rpm, 1),
            "operating_status": status
        })
        log_id += 1

df_sensor_logs = pd.DataFrame(sensor_logs)

# 4. Save out raw sensor files to our environment
df_inventory.to_csv("machinery_inventory.csv", index=False)
df_sensor_logs.to_csv("sensor_telemetry_logs.csv", index=False)

print("\nSUCCESS! Generated IoT Operational Layers.")
print(f"Machines Logged: {len(df_inventory)} | Total Telemetry Rows Extracted: {len(df_sensor_logs)}")

Initializing industrial environment simulation...
Generating minute-by-minute data for 50 machines. This creates ~2.1 million rows...

SUCCESS! Generated IoT Operational Layers.
Machines Logged: 50 | Total Telemetry Rows Extracted: 36000


## Phase 2: SQL Database & Window Analytics

In this phase, we will load our generated sensor data into a SQLite database to simulate a more realistic data storage environment. We will then use SQL queries, including window functions, to analyze the data.

In [2]:
import sqlite3

# Create a SQLite database in memory (or a file if you prefer)
conn = sqlite3.connect(':memory:') # Use ':memory:' for an in-memory database

# Write the pandas DataFrames to SQLite tables
df_inventory.to_sql('machinery_inventory', conn, if_exists='replace', index=False)
df_sensor_logs.to_sql('sensor_telemetry_logs', conn, if_exists='replace', index=False)

print("Data loaded into SQLite database successfully.")

Data loaded into SQLite database successfully.


Let's verify the tables and their contents by querying the database.

In [3]:
# Query the machinery_inventory table
print("\n--- Machinery Inventory (First 5 Rows) ---")
df_sql_inventory = pd.read_sql_query("SELECT * FROM machinery_inventory LIMIT 5;", conn)
display(df_sql_inventory)

# Query the sensor_telemetry_logs table
print("\n--- Sensor Telemetry Logs (First 5 Rows) ---")
df_sql_sensor_logs = pd.read_sql_query("SELECT * FROM sensor_telemetry_logs LIMIT 5;", conn)
display(df_sql_sensor_logs)


--- Machinery Inventory (First 5 Rows) ---


,machine_id,machine_type,factory_floor,installation_year
0,101,CNC Milling Machine,Floor A,2022
1,102,Industrial Robotic Arm,Floor A,2020
2,103,CNC Milling Machine,Floor C,2019
3,104,High-Speed Turbine,Floor A,2018
4,105,CNC Milling Machine,Floor A,2021



--- Sensor Telemetry Logs (First 5 Rows) ---


,log_id,machine_id,timestamp,temperature_celsius,vibration_mm_s,rotation_speed_rpm,operating_status
0,1,101,2026-06-01 00:00:00,51.24,1.459,1213.0,Normal
1,2,101,2026-06-01 01:00:00,53.81,1.430,1195.3,Normal
2,3,101,2026-06-01 02:00:00,53.95,1.730,1190.6,Normal
3,4,101,2026-06-01 03:00:00,51.36,1.361,1190.7,Normal
4,5,101,2026-06-01 04:00:00,50.60,0.926,1165.5,Normal


Now, let's explore **Window Analytics** in SQL. Window functions allow us to perform calculations across a set of table rows that are somehow related to the current row. This is particularly useful for time-series data to calculate moving averages, running totals, or compare values across time.

For example, we can calculate a 3-hour rolling average of `temperature_celsius` for each machine to identify trends or deviations.

In [4]:
# Example of a SQL Window Function: 3-hour rolling average of temperature for each machine
query = """
SELECT
    machine_id,
    timestamp,
    temperature_celsius,
    AVG(temperature_celsius) OVER (
        PARTITION BY machine_id
        ORDER BY timestamp
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS rolling_avg_temp_3hr
FROM
    sensor_telemetry_logs
ORDER BY
    machine_id, timestamp
LIMIT 20;
"""

print("\n--- 3-Hour Rolling Average Temperature (First 20 Rows) ---")
df_rolling_avg = pd.read_sql_query(query, conn)
display(df_rolling_avg)

# Close the connection (important for file-based DBs, less critical for in-memory)
# conn.close()


--- 3-Hour Rolling Average Temperature (First 20 Rows) ---


,machine_id,timestamp,temperature_celsius,rolling_avg_temp_3hr
0,101,2026-06-01 00:00:00,51.24,51.240000
1,101,2026-06-01 01:00:00,53.81,52.525000
2,101,2026-06-01 02:00:00,53.95,53.000000
3,101,2026-06-01 03:00:00,51.36,53.040000
4,101,2026-06-01 04:00:00,50.60,51.970000
5,101,2026-06-01 05:00:00,48.59,50.183333
6,101,2026-06-01 06:00:00,47.73,48.973333
7,101,2026-06-01 07:00:00,49.44,48.586667
8,101,2026-06-01 08:00:00,48.64,48.603333
9,101,2026-06-01 09:00:00,50.94,49.673333


## Phase 3: Stream Processing & Anomaly Detection

Now that we have our historical data in a database, let's simulate real-time data streams and apply anomaly detection. This is crucial for predictive maintenance, as it allows us to identify potential issues as they happen, rather than after a failure has occurred.

We will simulate data arriving in chunks (micro-batches) and process them to detect anomalies.

In [5]:
# For stream processing, we'll iterate through our sensor logs as if they were coming in real-time.
# Let's define a simple function to simulate a data stream.

def get_data_stream(df, chunk_size=100):
    for i in range(0, len(df), chunk_size):
        yield df.iloc[i : i + chunk_size]

# Initialize the data stream with our sensor logs
stream = get_data_stream(df_sensor_logs, chunk_size=1000)

print("Simulating data stream...")

Simulating data stream...


Next, let's implement a basic anomaly detection method. A common approach for time-series data is to use statistical methods, such as checking if a sensor reading deviates significantly from its recent average (e.g., beyond a certain number of standard deviations).

In [6]:
from sklearn.ensemble import IsolationForest

# Store detected anomalies
anomalies = []

# Let's process the first few chunks to demonstrate
num_chunks_to_process = 5

print(f"Processing {num_chunks_to_process} chunks for anomaly detection...")

for i, chunk_original in enumerate(stream):
    if i >= num_chunks_to_process:
        break

    print(f"Processing chunk {i+1}/{num_chunks_to_process}...")

    # Create a proper copy to avoid SettingWithCopyWarning
    chunk = chunk_original.copy()

    # For this example, we'll focus on 'temperature_celsius' and 'vibration_mm_s'
    # as features for anomaly detection within each chunk.
    features = chunk[['temperature_celsius', 'vibration_mm_s']]

    # Train an Isolation Forest model on the current chunk
    # Isolation Forest is good for anomaly detection as it's effective for high-dimensional datasets
    # and works well with new, unseen data, fitting our 'streaming' context.
    model = IsolationForest(random_state=42, contamination=0.01) # Assume 1% anomalies
    model.fit(features)

    # Predict anomalies (-1 for anomaly, 1 for normal)
    chunk['anomaly_score'] = model.decision_function(features)
    chunk['is_anomaly'] = model.predict(features)

    # Filter out actual anomalies
    detected_anomalies = chunk[chunk['is_anomaly'] == -1]
    if not detected_anomalies.empty:
        anomalies.append(detected_anomalies)
        print(f"  -> Detected {len(detected_anomalies)} anomalies in chunk {i+1}.")

if anomalies:
    df_anomalies = pd.concat(anomalies)
    print("\n--- Detected Anomalies (First 5) ---")
    display(df_anomalies.head())
else:
    print("\nNo anomalies detected in the processed chunks (this might be due to the small sample size or nature of generated data).")

Processing 5 chunks for anomaly detection...
Processing chunk 1/5...
  -> Detected 10 anomalies in chunk 1.
Processing chunk 2/5...
  -> Detected 10 anomalies in chunk 2.
Processing chunk 3/5...
  -> Detected 10 anomalies in chunk 3.
Processing chunk 4/5...
  -> Detected 10 anomalies in chunk 4.
Processing chunk 5/5...
  -> Detected 10 anomalies in chunk 5.

--- Detected Anomalies (First 5) ---


,log_id,machine_id,timestamp,temperature_celsius,vibration_mm_s,rotation_speed_rpm,operating_status,anomaly_score,is_anomaly
234,235,101,2026-06-10 18:00:00,83.24,5.577,467.5,Anomaly detected,-0.156989,-1
260,261,101,2026-06-11 20:00:00,85.44,5.599,483.0,Anomaly detected,-0.156989,-1
367,368,101,2026-06-16 07:00:00,42.76,2.127,1197.2,Normal,-0.047169,-1
513,514,101,2026-06-22 09:00:00,42.82,1.492,1235.4,Normal,-0.007996,-1
530,531,101,2026-06-23 02:00:00,47.32,0.624,1208.7,Normal,-0.017768,-1


The Isolation Forest model helps us identify data points that are significantly different from the majority. In a real-world scenario, these detected anomalies would trigger alerts for further investigation or automated actions.

We have successfully simulated a data stream and applied a basic anomaly detection algorithm. This lays the groundwork for real-time predictive maintenance.

## Phase 4: The Final Step — Enterprise Visual Preparation

Now that we have detected anomalies, the final step is to prepare a clear and concise summary for business intelligence dashboards. This involves aggregating the anomaly alerts, counting them by factory floor and machine type, and calculating relevant metrics. This will allow executives to quickly understand the state of their industrial assets.

In [7]:
import pandas as pd
import sqlite3

print("[Phase 4] Launching Enterprise Visualization Preparation Layer...")

# 1. Reconnect to our database engine and load the DataFrames from CSVs
conn = sqlite3.connect(":memory:") # Reusing in-memory database

# Load DataFrames from saved CSV files to ensure they are defined
df_inventory = pd.read_csv("machinery_inventory.csv")
df_sensor_logs = pd.read_csv("sensor_telemetry_logs.csv")

# Write the pandas DataFrames to SQLite tables
df_inventory.to_sql('machinery_inventory', conn, if_exists='replace', index=False)
df_sensor_logs.to_sql('sensor_telemetry_logs', conn, if_exists='replace', index=False)

# NOTE: The original script expected 'telemetry_logs' but our generated table is 'sensor_telemetry_logs'.
# Also, the 'operating_status' column would need to be updated with the detected anomalies.
# For demonstration, I'm adapting the query to use the existing 'operating_status' from df_sensor_logs
# and the correct table name.
# In a real scenario, the 'df_anomalies' from Phase 3 would be merged back or stored.

# 2. Write an advanced SQL query to aggregate anomalies by factory floor
summary_query = """
SELECT
    i.factory_floor,
    i.machine_type,
    COUNT(t.log_id) AS total_monitored_hours,
    SUM(CASE WHEN t.operating_status = 'Anomaly detected' THEN 1 ELSE 0 END) AS total_anomalies_detected,
    ROUND(AVG(t.temperature_celsius), 2) AS running_avg_temperature
FROM sensor_telemetry_logs t
JOIN machinery_inventory i ON t.machine_id = i.machine_id
GROUP BY i.factory_floor, i.machine_type;
"""

df_executive_summary = pd.read_sql_query(summary_query, conn)

# 3. Save the final corporate spreadsheet asset
df_executive_summary.to_csv("executive_machinery_summary.csv", index=False)

conn.close() # Close connection after all operations

print("\n==================================================")
print("SUCCESS! PROJECT COMPLETED.")
print("Final Summary View Generated for Dashboard:")
print("==================================================")
print(df_executive_summary.to_string(index=False))

[Phase 4] Launching Enterprise Visualization Preparation Layer...

SUCCESS! PROJECT COMPLETED.
Final Summary View Generated for Dashboard:
factory_floor           machine_type  total_monitored_hours  total_anomalies_detected  running_avg_temperature
      Floor A    CNC Milling Machine                   3600                         4                    50.13
      Floor A  Heavy Duty Compressor                   5760                        11                    65.10
      Floor A     High-Speed Turbine                   3600                        11                    75.12
      Floor A Industrial Robotic Arm                   2160                         4                    49.99
      Floor B    CNC Milling Machine                   2880                         9                    50.05
      Floor B  Heavy Duty Compressor                    720                         3                    65.08
      Floor B     High-Speed Turbine                   2160                         

In [8]:
# 1. Install Streamlit
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 104.6 MB/s eta 0:00:00


In [9]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

st.set_page_config(page_title="ApexSync Dashboard", layout="wide")
st.title("🏭 ApexSync: Industrial IoT Predictive Maintenance Control Center")
st.markdown("Real-time telemetry analysis and machine learning anomaly monitoring.")

# Load the data we prepared in Phase 4
df = pd.read_csv("executive_machinery_summary.csv")

# Metrics Display Row
col1, col2, col3 = st.columns(3)
col1.metric("Total Monitored Hours", f"{df['total_monitored_hours'].sum():,}")
col2.metric("Total Anomalies Detected", f"{df['total_anomalies_detected'].sum()}")
col3.metric("System Health Index", f"{round((1 - (df['total_anomalies_detected'].sum() / df['total_monitored_hours'].sum())) * 100, 2)}%")

st.markdown("---")

# Visualizations Row
left_chart, right_chart = st.columns(2)

with left_chart:
    st.subheader("⚠️ Anomalies Detected by Factory Floor")
    fig, ax = plt.subplots()
    floor_data = df.groupby("factory_floor")["total_anomalies_detected"].sum().reset_index()
    # Fix for Seaborn FutureWarning: Pass `x` to `hue` and set `legend=False`
    sns.barplot(data=floor_data, x="factory_floor", y="total_anomalies_detected", hue="factory_floor", legend=False, palette="Oranges_r", ax=ax)
    st.pyplot(fig)

with right_chart:
    st.subheader("🌡️ Running Average Temperature by Machine Type")
    fig, ax = plt.subplots()
    sns.heatmap(df.pivot(index="machine_type", columns="factory_floor", values="running_avg_temperature"),
                annot=True, fmt=".1f", cmap="YlOrRd", ax=ax)
    st.pyplot(fig)

# Data Table Display
st.subheader("📋 Raw Analytical Data Layer")
# Fix for Streamlit deprecation warning: Replace `use_container_width=True` with `width='stretch'`
st.dataframe(df, width='stretch')

Writing app.py


### Run the Streamlit Dashboard

To view your interactive dashboard, execute the following command in a new code cell. This will launch the Streamlit app and provide a public URL for you to access it.

First, we need to install `npx localtunnel` to expose the Streamlit server.


In [10]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 2s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

In [13]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴your url is: https://ten-icons-camp.loca.lt
2026-07-02 10:14:09.062 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.182.235.10:8501

  Stopping...
^C


In [14]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

st.set_page_config(page_title="ApexSync Dashboard", layout="wide")
st.title("🏭 ApexSync: Industrial IoT Predictive Maintenance Control Center")
st.markdown("Real-time telemetry analysis and machine learning anomaly monitoring.")

# Load the data we prepared in Phase 4
df = pd.read_csv("executive_machinery_summary.csv")

# Explicitly convert relevant columns to numeric types to prevent potential TypeErrors
df['total_monitored_hours'] = pd.to_numeric(df['total_monitored_hours'], errors='coerce')
df['total_anomalies_detected'] = pd.to_numeric(df['total_anomalies_detected'], errors='coerce')
df['running_avg_temperature'] = pd.to_numeric(df['running_avg_temperature'], errors='coerce')

# Drop any rows where conversion might have failed (though previous check indicates no NaNs)
df.dropna(subset=['total_monitored_hours', 'total_anomalies_detected', 'running_avg_temperature'], inplace=True)


# Metrics Display Row
col1, col2, col3 = st.columns(3)
col1.metric("Total Monitored Hours", f"{df['total_monitored_hours'].sum():,}")
col2.metric("Total Anomalies Detected", f"{df['total_anomalies_detected'].sum()}")
col3.metric("System Health Index", f"{round((1 - (df['total_anomalies_detected'].sum() / df['total_monitored_hours'].sum())) * 100, 2)}%")

st.markdown("---")

# Visualizations Row
left_chart, right_chart = st.columns(2)

with left_chart:
    st.subheader("⚠️ Anomalies Detected by Factory Floor")
    fig, ax = plt.subplots()
    floor_data = df.groupby("factory_floor")["total_anomalies_detected"].sum().reset_index()
    # Fix for Seaborn FutureWarning: Pass `x` to `hue` and set `legend=False`
    sns.barplot(data=floor_data, x="factory_floor", y="total_anomalies_detected", hue="factory_floor", legend=False, palette="Oranges_r", ax=ax)
    st.pyplot(fig)

with right_chart:
    st.subheader("🌡️ Running Average Temperature by Machine Type")
    fig, ax = plt.subplots()
    # Using pivot_table for robustness in case of unexpected data structure
    heatmap_data = df.pivot_table(index="machine_type", columns="factory_floor", values="running_avg_temperature")
    sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax)
    st.pyplot(fig)

# Data Table Display
st.subheader("📋 Raw Analytical Data Layer")
# Fix for Streamlit deprecation warning: Replace `use_container_width=True` with `width='stretch'`
st.dataframe(df, width='stretch')


Overwriting app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦your url is: https://small-eagles-behave.loca.lt
2026-07-02 10:17:49.487 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.182.235.10:8501

